In [ ]:
#note bbokk explores syudents proformances

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#First - data exploration  --- Task1

df = pd.read_csv(r"C:\Users\swaro\Downloads\ass3\students.csv")

df = pd.DataFrame(data)

df.head()


df.head()

print("Shape:", df.shape)
print("Data types:\n", df.dtypes)

df.describe()

#toal count of pass or fail data
df['passed'].value_counts()

#cal avg scores
subject_cols = ['math','science','english','history','pe']
print("Pass avg:\n", df[df['passed']==1][subject_cols].mean())
print("Fail avg:\n", df[df['passed']==0][subject_cols].mean())

df['overall_avg'] = df[subject_cols].mean(axis=1)
best_student = df.loc[df['overall_avg'].idxmax()]
print("Top student:", best_student['name'], "with avg", best_student['overall_avg'])

Shape: (15, 9)
Data types:
 name                    object
math                     int64
science                  int64
english                  int64
history                  int64
pe                       int64
attendance_pct           int64
study_hours_per_day    float64
passed                   int64
dtype: object
Pass avg:
 math       78.222222
science    78.555556
english    79.111111
history    73.444444
pe         86.000000
dtype: float64
Fail avg:
 math       45.166667
science    49.000000
english    46.833333
history    48.333333
pe         58.000000
dtype: float64
Top student: Diana with avg 94.0

#=========================================================================
#Second - viz

df['avg_score'] = df[subject_cols].mean(axis=1)

#bar chart
plt.bar(subject_cols, df[subject_cols].mean())
plt.title("Average Score per Subject")
plt.xlabel("Subject")
plt.ylabel("Average Score")
plt.show()


#histogram
plt.hist(df['math'], bins=5, color='skyblue', edgecolor='black')
plt.axvline(df['math'].mean(), color='red', linestyle='--', label='Mean')
plt.title("Distribution of Math Scores")
plt.xlabel("Math Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()


#scatter
plt.scatter(df[df['passed']==1]['study_hours_per_day'], df[df['passed']==1]['avg_score'], color='green', label='Pass')
plt.scatter(df[df['passed']==0]['study_hours_per_day'], df[df['passed']==0]['avg_score'], color='red', label='Fail')
plt.title("Study Hours vs Avg Score")
plt.xlabel("Study Hours per Day")
plt.ylabel("Average Score")
plt.legend()
plt.show()


#chartBox
pass_attendance = df[df['passed']==1]['attendance_pct'].tolist()
fail_attendance = df[df['passed']==0]['attendance_pct'].tolist()
plt.boxplot([pass_attendance, fail_attendance], labels=['Pass','Fail'])
plt.title("Attendance Distribution by Outcome")
plt.ylabel("Attendance %")
plt.show()


#linechart
plt.plot(df['name'], df['math'], marker='o', label='Math')
plt.plot(df['name'], df['science'], marker='s', label='Science')
plt.xticks(rotation=45)
plt.title("Math & Science Scores by Student")
plt.xlabel("Student")
plt.ylabel("Score")
plt.legend()
plt.show()



#=========================================================================
#third- seaborne viz

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(10,5))

sns.barplot(data=df, x='passed', y='math', ax=ax1)
ax1.set_title("Math Score by Pass/Fail")

sns.barplot(data=df, x='passed', y='science', ax=ax2)
ax2.set_title("Science Score by Pass/Fail")

plt.show()

# Scatter + regression
sns.regplot(data=df[df['passed']==1], x='attendance_pct', y='avg_score', label='Pass', scatter=True, color='green')
sns.regplot(data=df[df['passed']==0], x='attendance_pct', y='avg_score', label='Fail', scatter=True, color='red')
plt.title("Attendance vs Avg Score with Regression")
plt.xlabel("Attendance %")
plt.ylabel("Average Score")
plt.legend()
plt.show()

# Comment:
# Seaborn made the bar plots and regression scatter much easier by less code and styling it automatically.
# Matplotlib gave more control but need to do manual work.



#=========================================================================
#ml- fourth

X = df[['math','science','english','history','pe','attendance_pct','study_hours_per_day']]
y = df['passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("Training accuracy:", model.score(X_train_scaled, y_train))
print("Test accuracy:", model.score(X_test_scaled, y_test))

# Predictions with names
y_pred = model.predict(X_test_scaled)
for idx, pred in zip(X_test.index, y_pred):
    actual = y.loc[idx]
    name = df.loc[idx, 'name']
    correct = ":white_check_mark:" if actual==pred else ":x:"
    print(f"{name}: Actual={actual}, Predicted={pred} {correct}")

coeffs = model.coef_[0]
features = X.columns
importance = sorted(zip(features, coeffs), key=lambda x: abs(x[1]), reverse=True)
print("Feature importance :")
for f,c in importance:
    print(f"{f}: {c:.3f}")

colors = ['green' if c>0 else 'red' for c in coeffs]
plt.barh(features, coeffs, color=colors)
plt.title("Feature Coefficients (Logistic Regression)")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

#predicat new studentts - BONUS
new_student = [[75,70,68,65,80,82,3.2]]
new_student_scaled = scaler.transform(new_student)
pred = model.predict(new_student_scaled)[0]
prob = model.predict_proba(new_student_scaled)[0]
print("New student prediction:", "Pass" if pred==1 else "Fail")
print("Probabilities:", prob)

Training accuracy: 1.0
Test accuracy: 1.0
Jack: Actual=0, Predicted=0 :white_check_mark:
Liam: Actual=0, Predicted=0 :white_check_mark:
Alice: Actual=1, Predicted=1 :white_check_mark:
Feature importance :
english: 0.813
attendance_pct: 0.522
study_hours_per_day: 0.484
pe: 0.475
math: 0.438
science: 0.323
history: 0.263

New student prediction: Pass
Probabilities: [0.09203526 0.90796474]

#=========================================================================